# V2 stage-one acceptance and opt-in 300-minute paired experiment
Select T4 and mount Drive manually. Default does not run the full dataset or train production models. Never point outputs at V1. The overnight switch requires separate approval after acceptance. No keep-alive scripts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
REPO = Path('/content/IGCRN-DualMic-Speech-Enhancement')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/ZhengchenYuan/IGCRN-DualMic-Speech-Enhancement.git',str(REPO)],cwd='/content',check=True)
os.chdir(REPO)
RUN_OVERNIGHT = False  # Change only after stage-one acceptance and explicit approval.
RUN_ID = 'v2_overnight_001'  # Keep unchanged to resume; new name for a new experiment.
SAVE = Path('/content/drive/MyDrive/IGCRN_NIO/v2') / RUN_ID
DATA = Path('/content/igcrn_v2_3000')
CACHE = Path('/content/igcrn_v2_rir_cache')
if RUN_OVERNIGHT:
    # Persist deadline BEFORE installation/preparation. Resume never restarts this clock.
    SAVE.mkdir(parents=True,exist_ok=True)
    budget_path = SAVE / 'budget.json'
    if not budget_path.exists():
        now=time.time()
        budget_path.write_text(json.dumps(dict(schema_version=2,budget_minutes=300,started_unix=now,deadline_unix=now+18000)))
def run(*args,cpu=False):
    env=dict(os.environ)
    if cpu: env['CUDA_VISIBLE_DEVICES']=''
    result=subprocess.run([sys.executable,*map(str,args)],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(result.stdout)
    if RUN_OVERNIGHT:
        with (SAVE/'preflight.log').open('a') as log: log.write(str(args)+'\n'+result.stdout+'\n')
    result.check_returncode()
run('-m','pip','install','-r','requirements-v2.txt')
run('-m','pip','install','-e','.')


## Paths to real V1 evidence
Restore V1 train/valid/test manifests and actual source files. Preserve their original bytes and paths; do not substitute the exported mono Mic1 WAV for a dual-channel mixture. A test-only manifest cannot establish V1 training/validation identity protection. Adjust these paths before running.

In [ ]:
V1_DATA = Path('/content/igcrn_experiment_v2')
V1_DUAL = Path('/content/drive/MyDrive/IGCRN_NIO/dual/last.pt')
V1_MANIFESTS = [V1_DATA/f'{s}.jsonl' for s in ('train','valid','test')]
CANDIDATE_CATALOG = Path('/content/igcrn_candidates/catalog.jsonl')
REGISTRY = Path('/content/igcrn_v2_registry.json')
STAGE = Path('/content/igcrn_v2_stage1')
STAGE.mkdir(exist_ok=True)


## Small physical acceptance, CPU only
Each command needs a fresh output path. Full validation/test audio is preferable for V1 diagnostics. Missing source waveforms are reported UNVERIFIED, never inferred from mono exports. The acoustic probe checks direct-delay precision and reports measured decay, not just requested RT60.

In [ ]:
if not (STAGE/'acoustics').exists():
    run('-m','scripts.accept_acoustics','--output',STAGE/'acoustics',cpu=True)
physical=json.loads((STAGE/'acoustics'/'acoustics.json').read_text())
assert physical['status']=='PASS', physical['status']
if V1_MANIFESTS[1].exists() and V1_DUAL.exists() and not (STAGE/'v1').exists():
    run('-m','scripts.diagnose_spatial','--manifest',V1_MANIFESTS[1],'--dual',V1_DUAL,'--output',STAGE/'v1',cpu=True)
else:
    print('V1 diagnostics not newly run: restore paths or inspect existing stage output.')


## Candidate source pool and fixed split registry
If the candidate catalog does not exist, explicitly enable public preparation below. Uses the small Mini LibriSpeech archive and selected ESC-10 recordings, never train-clean-100. The registry overrides candidate split labels with locked V1 identities. Inspect actual source inventory after registration.

In [ ]:
PREPARE_PUBLIC_SOURCES = False
if not CANDIDATE_CATALOG.exists() and PREPARE_PUBLIC_SOURCES:
    run('-m','scripts.prepare_data','--mode','experiment','--output',CANDIDATE_CATALOG.parent,
        '--speakers',24,6,6,'--utterances-per-speaker',40,'--noise-recordings',80,20,20,cpu=True)
assert all(p.exists() for p in V1_MANIFESTS), 'Restore all THREE original V1 manifests first'
assert CANDIDATE_CATALOG.exists(), 'Prepare/restore the verified candidate catalog and source bytes'
if not REGISTRY.exists():
    run('-m','scripts.register_v2_split','--v1-manifests',*V1_MANIFESTS,'--catalog',CANDIDATE_CATALOG,'--v1-dual',V1_DUAL,'--output',REGISTRY,cpu=True)


In [ ]:
# Acceptance fixture: 3 real-source samples per split, clearly labeled non-production.
if not (STAGE/'small_data').exists():
    run('-m','scripts.simulate_v2','--registry',REGISTRY,'--output',STAGE/'small_data',
        '--cache',CACHE,'--acceptance-count',3,cpu=True)
run('-m','pytest','-q','tests/v2',cpu=True)  # Tiny synthetic optimizer tests, not production training.


## Approval gate: complete overnight controller
When approved, set RUN_OVERNIGHT=True above and run in order. The timer includes installation, diagnosis and preparation. Both models alternate whole epochs and only a verified pair is published. Incomplete epochs are discarded on restart. Evaluation remains CPU to preserve the existing protocol. A missing COMPLETE marker is not a completed experiment.

In [ ]:
if RUN_OVERNIGHT:
    import shutil, yaml
    from scripts.run_v2 import initialize_run
    initialize_run(yaml.safe_load((REPO/'configs/v2.yaml').read_text()),DATA,SAVE)
    SAVE.mkdir(parents=True,exist_ok=True)
    shutil.copyfile(REGISTRY,SAVE/'registry.json')
    shutil.copytree(STAGE/'acoustics',SAVE/'acoustic_acceptance',dirs_exist_ok=True)
    if (STAGE/'v1').exists(): shutil.copytree(STAGE/'v1',SAVE/'v1_diagnostic',dirs_exist_ok=True)
    with (SAVE/'environment.txt').open('w') as stream:
        subprocess.run([sys.executable,'-m','pip','freeze'],stdout=stream,cwd=REPO,check=True)
    run('-m','scripts.run_v2','--data',DATA,'--registry',REGISTRY,'--cache',CACHE,'--run',SAVE,'--execute')
    print((SAVE/'state.json').read_text())
    print('Complete:',(SAVE/'experiment_complete.json').exists())
else:
    print('Stage one only. Production dataset/training/evaluation were NOT launched.')
